# Quantum Computing: Error Correction & Fault Tolerance

## Surface Codes for Fault-Tolerant Quantum Computing

**Breakthrough** (Kitaev, 2003): Topological codes enable threshold for fault tolerance  
**Problem**: Quantum errors accumulate; each gate ~0.1% error rate  
**Solution**: Quantum error correction via stabilizer codes

### The Threshold
If physical error rate ε < ε_th ≈ 10^{-3}, logical error decays exponentially:
```
ε_L ~ (ε/ε_th)^{(d+1)/2}
```
where d is code distance (space overhead).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Surface Code Architecture

### 2D Grid of Physical Qubits
```
● ─ ● ─ ● ─ ●
│   │   │   │
● ─ ● ─ ● ─ ●
│   │   │   │
● ─ ● ─ ● ─ ●
```

### Stabilizer Operators
- **Plaquette (Z-type)**: Z₁Z₂Z₃Z₄ (measure parity of 4-body)
- **Star (X-type)**: X₁X₂X₃X₄

Eigenvalues: ±1 encode syndrome (error location)

### Error Correction Process
1. Measure all stabilizers (non-destructive)
2. Extract syndrome pattern
3. Decode: σ = arg min d(syndrome, S(error))
4. Apply correction


In [ ]:
class SurfaceCodeSimulator:
    """Simulates surface code error correction."""
    
    def __init__(self, distance):
        """Initialize surface code.
        
        distance: code distance (d × d grid)
        """
        self.d = distance
        self.physical_qubits = distance ** 2
        self.logical_qubits = 1
        
        # Error threshold
        self.error_threshold = 0.01
    
    def space_overhead(self):
        """Number of physical qubits per logical qubit."""
        return self.physical_qubits / self.logical_qubits
    
    def logical_error_rate(self, physical_error_rate):
        """Logical error rate after correction.
        
        If ε < ε_th: ε_L ~ (ε/ε_th)^{(d+1)/2}
        If ε ≥ ε_th: ε_L ~ ε (no advantage)
        """
        if physical_error_rate < self.error_threshold:
            ratio = physical_error_rate / self.error_threshold
            exponent = (self.d + 1) / 2
            return ratio ** exponent
        else:
            return physical_error_rate
    
    def summary(self):
        return {
            'distance': self.d,
            'physical_qubits': self.physical_qubits,
            'space_overhead': self.space_overhead(),
            'threshold': self.error_threshold
        }

# Test different code distances
print("SURFACE CODE ERROR CORRECTION")
print("="*70)
print(f"{'Distance':>10} {'Physical Qubits':>20} {'Space Overhead':>20}")
print("-"*70)
for d in [3, 5, 7, 9, 11]:
    code = SurfaceCodeSimulator(d)
    print(f"{d:>10} {code.physical_qubits:>20} {code.space_overhead():>20.0f}x")

print(f"\nNote: Space overhead grows as d² (quadratic)")
print(f"Time overhead: O(d) for syndrome decoding")

## Part 2: Threshold & Fault Tolerance


In [ ]:
# Plot logical error vs physical error
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Phase transition (threshold)
ax = axes[0]
physical_errors = np.logspace(-4, -1.5, 100)
distances = [3, 5, 7, 9]

for d in distances:
    code = SurfaceCodeSimulator(d)
    logical_errors = [code.logical_error_rate(pe) for pe in physical_errors]
    ax.loglog(physical_errors, logical_errors, 'o-', label=f'd={d}', linewidth=2, markersize=4)

ax.axvline(x=0.01, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Threshold ε_th')
ax.loglog(physical_errors, physical_errors, 'k--', linewidth=1, alpha=0.5, label='No correction (ε_L=ε)')
ax.set_xlabel('Physical Error Rate')
ax.set_ylabel('Logical Error Rate')
ax.set_title('Surface Code: Below Threshold, Logical Errors Decay')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim([1e-4, 1e-1.5])

# Plot 2: Overhead vs distance
ax = axes[1]
distances_range = np.arange(3, 16, 2)
overheads = [d**2 for d in distances_range]
thresholds = [0.01 * (0.1)**(d-1)/2 for d in distances_range]  # Mock decreasing threshold

ax2 = ax.twinx()
line1 = ax.plot(distances_range, overheads, 'o-', linewidth=2, markersize=8, 
                color='steelblue', label='Space Overhead')
line2 = ax2.semilogy(distances_range, thresholds, 's-', linewidth=2, markersize=8, 
                     color='darkgreen', label='Required Phys Error Rate')

ax.set_xlabel('Code Distance (d)')
ax.set_ylabel('Physical Qubits per Logical Qubit', color='steelblue')
ax2.set_ylabel('Required Physical Error Rate', color='darkgreen')
ax.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='darkgreen')
ax.set_title('Resource Requirements for Fault Tolerance')
ax.grid(True, alpha=0.3)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper left')

plt.tight_layout()
plt.savefig('SECTION_4_QUANTUM/error_correction.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHARDWARE PROGRESS TOWARDS THRESHOLD")
print("="*70)
print(f"Year        Hardware           Error Rate    Distance Required")
print("-"*70)
print(f"2024        Google Willow     ~10^-3         d~3 (approaching)")
print(f"2025-26     IBM/IonQ Improved ~10^-4         d~5-7 (feasible)")
print(f"2027+       Scaled Systems    <10^-5         d~11+ (large-scale)")
print(f"\nGoal: Error rate below 10^-3 and d large enough for practical computation")

## Part 3: Decoding & Matching


In [ ]:
# Minimum weight perfect matching (MWPM) decoder simulation
class SurfaceCodeDecoder:
    """MWPM decoder for surface codes."""
    
    def __init__(self, distance):
        self.d = distance
        self.stabilizers = (2*distance - 1) ** 2  # Number of stabilizer measurements
    
    def decode(self, syndrome, error_rate=0.01):
        """Decode syndrome → error correction.
        
        In practice: MWPM on graph of syndrome defects
        Here: mock simulation
        """
        success_prob = 1 - error_rate
        return np.random.rand() < success_prob

decoder = SurfaceCodeDecoder(distance=5)

# Simulate repeated rounds of error correction
error_rates = np.array([0.001, 0.003, 0.005, 0.010, 0.015])
success_probs = 1 - error_rates

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(error_rates * 100, success_probs * 100, 'o-', linewidth=2, markersize=10, color='darkblue')
ax.axvline(x=1.0, color='green', linestyle='--', linewidth=2, label='Typical current (1%)')
ax.axvline(x=0.1, color='orange', linestyle='--', linewidth=2, label='Goal for FTQC (0.1%)')
ax.set_xlabel('Physical Error Rate (%)')
ax.set_ylabel('Correction Success Rate (%)')
ax.set_title('Error Correction Success vs Physical Error Rate')
ax.set_ylim([90, 100.5])
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left')

plt.tight_layout()
plt.savefig('SECTION_4_QUANTUM/error_correction_decoder.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Insights

1. **Threshold theorem**: Below ε_th, error correction overhead pays off
2. **Exponential suppression**: Logical error rate decays exponentially with code distance
3. **Space-time tradeoff**: d² physical qubits, O(d) decoding time
4. **2025 milestone**: Google Willow approaches threshold for first time
5. **Roadmap**: 2026-2027: d=5-7 feasible; 2028+: large-scale FTQC possible

### References
- Kitaev, A. (2003). "Fault-Tolerant Quantum Computation by Anyons"
- Terhal, B. M. (2015). "Quantum error correction for quantum memories"
